# Exploración: SUNAT - Consulta RUC (Padrón de Contribuyentes)

- **Entidad oficial:** Superintendencia Nacional de Aduanas y de Administración Tributaria (SUNAT).
- **Descripción y propósito:** Registro tributario oficial que centraliza la identificación, condición legal, actividades económicas y estructura operativa de las personas naturales y jurídicas en el Perú. En el análisis de mercado y contrataciones públicas, permite auditar la existencia real de un proveedor, detectar estados irregulares (`BAJA DE OFICIO`, `NO HABIDO`) y cruzar domicilios fiscales, sucursales y representantes legales entre empresas competidoras.
- **Enlace de consulta:** `https://e-consultaruc.sunat.gob.pe/cl-ti-itmrconsruc/FrameCriterioBusquedaWeb.jsp`
- **Criterios de búsqueda soportados:**
  - **Por RUC (11 dígitos):** Acceso directo a la ficha integral del contribuyente y a sus submódulos societarios y operativos.
  - **Por Documento (DNI de 8 dígitos):** Búsqueda intermedia en la *"Relación de contribuyentes"* para identificar el RUC `10...` asociado a una persona natural y extraer su ficha descriptiva.
- **Información que se obtiene:**
  - **Ficha descriptiva general:** Número de RUC y Razón Social, Tipo de Contribuyente, Nombre Comercial, Fechas de Inscripción e Inicio de Actividades, Estado y Condición del Contribuyente, Domicilio Fiscal, Sistemas de Emisión/Contabilidad, Actividades Económicas (CIIU Principal y Secundarias), Comprobantes autorizados, Emisión Electrónica, afiliación al PLE, Padrones y Alertas Tributarias.
  - **Representantes Legales (Submódulo por RUC):** Tipo y número de documento de identidad, nombre completo, cargo societario (ej. `TITULAR-GERENTE`, `GERENTE GENERAL`) y fecha de inicio en el cargo (`Fecha Desde`).
  - **Establecimientos Anexos (Submódulo por RUC):** Código de local (`0001`, `0004`, etc.), tipo de establecimiento (`LO. L. COMERCIAL`, `DE. DEPOSITO`, etc.), dirección completa con ubigeo y código de actividad económica.

## Estrategia

- **Arquitectura del portal:** Aplicación web **Java EE / JSP sobre Oracle WebLogic** que procesa las vistas mediante peticiones `POST` hacia el servlet central `/cl-ti-itmrconsruc/jcrS00Alias`, protegida en el borde por **Cloudflare Turnstile / Anti-Bot**.
- **Evasión de bloqueos anti-bot (Stealth):**
  1. **Binario nativo del sistema:** Ejecución de `Playwright` invocando el navegador real instalado en el sistema operativo (`channel="chrome"` con fallback a `"msedge"`) y el argumento `--disable-blink-features=AutomationControlled` para mantener una huella TLS/HTTP2 legítima.
  2. **Ocultamiento de automatización:** Inyección previa mediante `add_init_script` para redefinir `navigator.webdriver = undefined` antes de cargar el DOM inicial.
- **Extracción de submódulos vía *In-Page Fetch* (sin recarga de página):**
  1. En las consultas por RUC, los botones inferiores (*"Representante(s) Legal(es)"* y *"Establecimiento(s) Anexo(s)"*) son formularios ocultos que disparan un `submit()` hacia `jcrS00Alias`. Navegar físicamente entre ellos activa el bloqueo `"Just a moment..."` de Cloudflare y destruye el contexto de la vista principal.
  2. Se aplica *monkey-patching* temporal sobre `HTMLFormElement.prototype.submit` dentro del navegador (`page.evaluate`) para interceptar el payload exacto (`FormData`) generado por el JavaScript nativo de SUNAT al presionar cada botón y enviarlo en segundo plano mediante `fetch()` asíncrono con decodificación `ISO-8859-1`.
- **Resolución de flujo dual (RUC vs. DNI):**
  - Para **RUC (11 dígitos)**, el formulario consulta `#txtRuc` y aterriza directamente en la ficha completa con acceso a los botones inferiores.
  - Para **DNI (8 dígitos)**, el flujo conmuta a la pestaña `#btnPorDocumento`, ingresa el DNI en `#txtNumeroDocumento`, captura el resumen previo en la vista intermedia *"Relación de contribuyentes"* (`a.aRuc`) y acciona el primer resultado para desplegar su ficha descriptiva.
- **Parseo estructurado (`BeautifulSoup`):**
  - Lectura de la grilla **Bootstrap** (`.list-group-item .row`) discriminando filas de 2 columnas (etiqueta-valor o subtablas de actividades/comprobantes) y 4 columnas (fechas y sistemas), filtrando cualquier contenedor que incluya etiquetas `<button>`, `<form>` o `<input>` para no contaminar el diccionario con controles de la interfaz.

In [4]:
import asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import re
import json

# Configuración exclusiva para consulta por RUC (11 dígitos)
RUC_PRUEBA = "20607524476"
URL_SUNAT = "https://e-consultaruc.sunat.gob.pe/cl-ti-itmrconsruc/FrameCriterioBusquedaWeb.jsp"

def limpiar(texto: str) -> str:
    if not texto:
        return ""
    return re.sub(r"\s+", " ", texto).strip()

def extraer_ficha_descriptiva(html: str) -> dict:
    soup = BeautifulSoup(html, "html.parser")
    ficha = {}

    # 1. Capturar alerta tributaria si existe (ej: BAJA DE OFICIO)
    alerta = soup.select_one(".list-group-item-danger")
    if alerta:
        ficha["Alerta Tributaria"] = limpiar(alerta.get_text())

    # 2. Recorrer únicamente las filas descriptivas (ignorando contenedores con botones, forms o inputs)
    for item in soup.select(".list-group-item"):
        if item.find(["button", "form", "input"]):
            continue

        for row in item.select(".row"):
            cols = row.find_all("div", recursive=False)

            # Filas de 4 columnas (ej: Fecha de Inscripción + Fecha de Inicio, Sistema Emisión + Comercio Exterior)
            if len(cols) == 4:
                t1, v1 = limpiar(cols[0].get_text()), limpiar(cols[1].get_text())
                t2, v2 = limpiar(cols[2].get_text()), limpiar(cols[3].get_text())
                if t1.endswith(":"):
                    ficha[t1.rstrip(":")] = v1
                if t2.endswith(":"):
                    ficha[t2.rstrip(":")] = v2

            # Filas de 2 columnas (Etiqueta + Valor simple o Tabla interna)
            elif len(cols) >= 2:
                t1 = limpiar(cols[0].get_text())
                if not t1.endswith(":"):
                    continue
                k = t1.rstrip(":")

                tabla_interna = cols[1].find("table")
                if tabla_interna:
                    lineas = [limpiar(tr.get_text()) for tr in tabla_interna.find_all("tr") if limpiar(tr.get_text())]
                    ficha[k] = lineas if len(lineas) > 1 else (lineas[0] if lineas else "-")
                else:
                    ficha[k] = limpiar(cols[1].get_text())

    return ficha

def parse_representantes(html: str) -> list[dict]:
    soup = BeautifulSoup(html, "html.parser")
    reps = []
    for tr in soup.find_all("tr"):
        cols = [limpiar(td.get_text()) for td in tr.find_all("td")]
        # Validar las 5 columnas (Documento, Nro, Nombre, Cargo, Fecha Desde)
        if len(cols) >= 5 and not cols[0].lower().startswith("documento"):
            reps.append({
                "document_type": cols[0],
                "document_number": cols[1],
                "full_name": cols[2],
                "role": cols[3],
                "start_date": cols[4]
            })
    return reps

def parse_establecimientos(html: str) -> list[dict]:
    soup = BeautifulSoup(html, "html.parser")
    anexos = []
    for tr in soup.find_all("tr"):
        cols = [limpiar(td.get_text(" ", strip=True)) for td in tr.find_all("td")]
        # Validar las 4 columnas (Código, Tipo, Dirección, Actividad Económica)
        if len(cols) >= 4 and not cols[0].lower().startswith(("código", "codigo")):
            anexos.append({
                "code": cols[0],
                "establishment_type": cols[1],
                "address": cols[2],
                "economic_activity": cols[3]
            })
    return anexos

async def consultar_sunat_completo(ruc: str) -> dict:
    ruc_limpio = ruc.strip()
    if len(ruc_limpio) != 11 or not ruc_limpio.isdigit():
        return {"error": "La consulta consolidada de SUNAT requiere exclusivamente un RUC numérico de 11 dígitos."}

    async with async_playwright() as p:
        try:
            browser = await p.chromium.launch(
                headless=True,
                channel="chrome",
                args=["--disable-blink-features=AutomationControlled"]
            )
        except Exception:
            browser = await p.chromium.launch(
                headless=True,
                channel="msedge",
                args=["--disable-blink-features=AutomationControlled"]
            )

        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
            viewport={"width": 1366, "height": 768},
            locale="es-PE"
        )
        await context.add_init_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
        page = await context.new_page()

        print(f"[1/4] Conectando a SUNAT y consultando RUC: {ruc_limpio}...")
        await page.goto(URL_SUNAT, wait_until="domcontentloaded", timeout=30000)
        await page.wait_for_selector("#txtRuc", timeout=10000)
        await page.fill("#txtRuc", ruc_limpio)
        await page.click("#btnAceptar")

        print("[2/4] Esperando carga de la ficha principal en jcrS00Alias...")
        await page.wait_for_selector("text=Número de RUC:", timeout=15000)

        html_ficha = await page.content()
        ficha_descriptiva = extraer_ficha_descriptiva(html_ficha)

        # Script JS para interceptar el POST de los botones inferiores en memoria sin recargar la página
        js_extraer_boton = """
        async (textoBoton) => {
            const botones = Array.from(document.querySelectorAll('button, input[type="button"], input[type="submit"]'));
            const btn = botones.find(b => (b.innerText || b.value || '').includes(textoBoton));
            if (!btn) return "";

            return new Promise((resolve) => {
                const origSubmit = HTMLFormElement.prototype.submit;
                
                HTMLFormElement.prototype.submit = function() {
                    const formData = new URLSearchParams(new FormData(this)).toString();
                    const actionUrl = this.action || 'jcrS00Alias';
                    HTMLFormElement.prototype.submit = origSubmit;
                    
                    fetch(actionUrl, {
                        method: 'POST',
                        headers: { 'Content-Type': 'application/x-www-form-urlencoded' },
                        body: formData
                    })
                    .then(r => r.arrayBuffer())
                    .then(buf => {
                        const html = new TextDecoder('iso-8859-1').decode(buf);
                        resolve(html);
                    })
                    .catch(() => resolve(""));
                };

                btn.click();

                setTimeout(() => {
                    HTMLFormElement.prototype.submit = origSubmit;
                    resolve("");
                }, 5000);
            });
        }
        """

        print("[3/4] Extrayendo submódulos (Representantes Legales y Establecimientos Anexos)...")
        html_reps = await page.evaluate(js_extraer_boton, "Representante")
        html_anexos = await page.evaluate(js_extraer_boton, "Establecimiento")

        representantes = parse_representantes(html_reps)
        establecimientos = parse_establecimientos(html_anexos)

        print(f"[4/4] ¡Extracción completada! Ficha: {len(ficha_descriptiva)} campos | Reps: {len(representantes)} | Anexos: {len(establecimientos)}")
        await browser.close()

        return {
            "general_info": ficha_descriptiva,
            "legal_representatives": representantes,
            "branch_offices": establecimientos
        }

# Ejecución en el notebook
resultado_sunat = await consultar_sunat_completo(RUC_PRUEBA)

print("\n================== 1. FICHA DESCRIPTIVA ==================")
for k, v in resultado_sunat.get("general_info", {}).items():
    print(f"- {k}: {json.dumps(v, ensure_ascii=False) if isinstance(v, list) else v}")

print("\n================== 2. REPRESENTANTES LEGALES ==================")
for r in resultado_sunat.get("legal_representatives", []):
    print(json.dumps(r, ensure_ascii=False))

print("\n================== 3. ESTABLECIMIENTOS ANEXOS ==================")
for b in resultado_sunat.get("branch_offices", []):
    print(json.dumps(b, ensure_ascii=False))

[1/4] Conectando a SUNAT y consultando RUC: 20607524476...
[2/4] Esperando carga de la ficha principal en jcrS00Alias...
[3/4] Extrayendo submódulos (Representantes Legales y Establecimientos Anexos)...
[4/4] ¡Extracción completada! Ficha: 19 campos | Reps: 1 | Anexos: 2

================== 1. FICHA DESCRIPTIVA ==================
- Alerta Tributaria: IMPORTANTE: Los comprobantes de pago o notas de débito emitidos por este contribuyente no dan derecho a crédito fiscal del IGV, en tanto se encuentra con estado de BAJA DE OFICIO
- Número de RUC: 20607524476 - INVERSIONES Y SERVICIOS HUAYNA DEL PERU EMPRESA INDIVIDUAL DE RESPONSABILIDAD LIMITADA
- Tipo Contribuyente: EMPRESA INDIVIDUAL DE RESP. LTDA
- Nombre Comercial: INVERSIONES HUAYNA DEL PERU
- Fecha de Inscripción: 24/02/2021
- Fecha de Inicio de Actividades: 15/03/2021
- Estado del Contribuyente: BAJA DE OFICIO Fecha de Baja: 30/04/2026
- Condición del Contribuyente: HABIDO
- Domicilio Fiscal: CAL.VILLA LA RINCONADA LOTE. H-1 APV. VI

In [5]:
import asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import re
import json

# Configuración exclusiva para consulta por DNI (8 dígitos)
DNI_PRUEBA = "44900952"
URL_SUNAT = "https://e-consultaruc.sunat.gob.pe/cl-ti-itmrconsruc/FrameCriterioBusquedaWeb.jsp"

def limpiar(texto: str) -> str:
    if not texto:
        return ""
    return re.sub(r"\s+", " ", texto).strip()

def extraer_ficha_descriptiva(html: str) -> dict:
    soup = BeautifulSoup(html, "html.parser")
    ficha = {}

    # 1. Capturar alerta tributaria si existe
    alerta = soup.select_one(".list-group-item-danger")
    if alerta:
        ficha["Alerta Tributaria"] = limpiar(alerta.get_text())

    # 2. Recorrer las filas descriptivas ignorando botones, formularios o inputs
    for item in soup.select(".list-group-item"):
        if item.find(["button", "form", "input"]):
            continue

        for row in item.select(".row"):
            cols = row.find_all("div", recursive=False)

            # Filas de 4 columnas (ej: Fecha de Inscripción + Fecha de Inicio, Sistema Emisión + Comercio Exterior)
            if len(cols) == 4:
                t1, v1 = limpiar(cols[0].get_text()), limpiar(cols[1].get_text())
                t2, v2 = limpiar(cols[2].get_text()), limpiar(cols[3].get_text())
                if t1.endswith(":"):
                    ficha[t1.rstrip(":")] = v1
                if t2.endswith(":"):
                    ficha[t2.rstrip(":")] = v2

            # Filas de 2 columnas (Etiqueta + Valor simple o Tabla interna)
            elif len(cols) >= 2:
                t1 = limpiar(cols[0].get_text())
                if not t1.endswith(":"):
                    continue
                k = t1.rstrip(":")

                tabla_interna = cols[1].find("table")
                if tabla_interna:
                    lineas = [limpiar(tr.get_text()) for tr in tabla_interna.find_all("tr") if limpiar(tr.get_text())]
                    ficha[k] = lineas if len(lineas) > 1 else (lineas[0] if lineas else "-")
                else:
                    ficha[k] = limpiar(cols[1].get_text())

    return ficha

async def consultar_sunat_por_dni(dni: str) -> dict:
    dni_limpio = dni.strip()
    if len(dni_limpio) != 8 or not dni_limpio.isdigit():
        return {"error": "La consulta por documento requiere exclusivamente un DNI numérico de 8 dígitos."}

    async with async_playwright() as p:
        try:
            browser = await p.chromium.launch(
                headless=True,
                channel="chrome",
                args=["--disable-blink-features=AutomationControlled"]
            )
        except Exception:
            browser = await p.chromium.launch(
                headless=True,
                channel="msedge",
                args=["--disable-blink-features=AutomationControlled"]
            )

        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
            viewport={"width": 1366, "height": 768},
            locale="es-PE"
        )
        await context.add_init_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
        page = await context.new_page()

        print(f"[1/4] Conectando a SUNAT para consultar DNI: {dni_limpio}...")
        await page.goto(URL_SUNAT, wait_until="domcontentloaded", timeout=30000)

        # Paso 1: Cambiar a la pestaña "Por Documento"
        print("[2/4] Seleccionando pestaña 'Por Documento' e ingresando DNI...")
        btn_doc = page.locator("#btnPorDocumento, button:has-text('Por Documento')").first
        await btn_doc.click()

        # El select #cmbTipoDoc ya viene por defecto en "Documento Nacional de Identidad" (valor "1")
        input_doc = page.locator("#txtNumeroDocumento, input[placeholder*='documento' i]").first
        await input_doc.wait_for(state="visible", timeout=10000)
        await input_doc.fill(dni_limpio)
        await page.click("#btnAceptar, button:has-text('Buscar')")

        # Paso 2: Esperar la vista intermedia "Relación de contribuyentes" y hacer clic en el primer resultado
        print("[3/4] Esperando 'Relación de contribuyentes' y abriendo el primer resultado...")
        item_resultado = page.locator("a.aRuc, .list-group-item.aRuc, .list-group a:has-text('RUC:')").first

        try:
            await item_resultado.wait_for(state="visible", timeout=15000)
        except Exception:
            # Verificar si el DNI no tiene RUC registrado en SUNAT
            texto_pagina = await page.inner_text("body")
            await browser.close()
            if "no se encontraron" in texto_pagina.lower() or "no existe" in texto_pagina.lower():
                return {"dni": dni_limpio, "registrado_en_sunat": False, "general_info": {}}
            return {"error": "No se cargó la relación de contribuyentes para el DNI indicado."}

        # Extraer brevemente los datos de la tarjeta previa (Ubicación / RUC detectado) antes del clic
        texto_tarjeta = limpiar(await item_resultado.inner_text())
        print(f" -> Contribuyente detectado en lista: {texto_tarjeta}")

        # Hacer clic en la tarjeta del contribuyente para abrir su ficha completa
        await item_resultado.click()

        # Paso 3: Esperar a que cargue la Ficha RUC completa ("Resultado de la Búsqueda")
        await page.wait_for_selector("text=Número de RUC:", timeout=15000)
        html_ficha = await page.content()
        ficha_descriptiva = extraer_ficha_descriptiva(html_ficha)

        print(f"[4/4] ¡Ficha por DNI extraída con éxito! ({len(ficha_descriptiva)} campos obtenidos).")
        await browser.close()

        return {
            "dni_consultado": dni_limpio,
            "resumen_relacion": texto_tarjeta,
            "general_info": ficha_descriptiva
        }

# Ejecución en el notebook
resultado_dni = await consultar_sunat_por_dni(DNI_PRUEBA)

print("\n================== FICHA SUNAT POR DNI ==================")
print(f"Resumen lista previa: {resultado_dni.get('resumen_relacion')}\n")
for k, v in resultado_dni.get("general_info", {}).items():
    print(f"- {k}: {json.dumps(v, ensure_ascii=False) if isinstance(v, list) else v}")

[1/4] Conectando a SUNAT para consultar DNI: 44900952...
[2/4] Seleccionando pestaña 'Por Documento' e ingresando DNI...
[3/4] Esperando 'Relación de contribuyentes' y abriendo el primer resultado...
 -> Contribuyente detectado en lista: RUC: 10449009521 RUIZ AUCCAPIÑA RONALD Ubicación: CUSCO Estado: ACTIVO
[4/4] ¡Ficha por DNI extraída con éxito! (19 campos obtenidos).

================== FICHA SUNAT POR DNI ==================
Resumen lista previa: RUC: 10449009521 RUIZ AUCCAPIÑA RONALD Ubicación: CUSCO Estado: ACTIVO

- Número de RUC: 10449009521 - RUIZ AUCCAPIÑA RONALD
- Tipo Contribuyente: PERSONA NATURAL SIN NEGOCIO
- Tipo de Documento: DNI 44900952 - RUIZ AUCCAPIÑA, RONALD
- Nombre Comercial: -
- Fecha de Inscripción: 30/01/2019
- Fecha de Inicio de Actividades: 30/01/2019
- Estado del Contribuyente: ACTIVO
- Condición del Contribuyente: HABIDO
- Domicilio Fiscal: -
- Sistema Emisión de Comprobante: COMPUTARIZADO
- Actividad Comercio Exterior: SIN ACTIVIDAD
- Sistema Contabilidad